# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itsayeshaqamar/flyrank-mlinternship-ayesha/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature Vector

The feature vector is built from observed search, traffic, AI-traffic, and engagement signals defined in the ML-04 data contract.

I use performance fields that describe the current or previously observed state of a content page. Client and content identifiers are not included as model features because they identify entities rather than describe performance.

For missing performance values, I use median imputation for numeric features. This is a simple baseline treatment, while data-availability flags are retained separately as context so missing measurements are not automatically interpreted as zero.

The resulting feature matrix contains only numeric performance features that can be used for the refresh-prioritization task.

In [3]:

import pandas as pd

import pandas as pd

# Load March 2026 daily performance data from the FlyRank warehouse
df = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())



Dataset shape: (9841378, 31)

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

First 5 rows:


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load the warehouse daily performance data

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "ai_chatgpt",
    "ai_perplexity",
    "ai_gemini",
    "ai_copilot",
    "ai_claude",
    "ai_meta",
    "ai_other",
    "scroll_events"
]

# Create the feature vector
X = df[feature_columns]

print("Number of feature fields:", len(feature_columns))
print("Feature vector shape:", X.shape)

print("\nFeature fields:")
print(feature_columns)

print("\nFeature vector preview:")
display(X.head())

Number of feature fields: 23
Feature vector shape: (9841378, 23)

Feature fields:
['gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

Feature vector preview:


,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,ga4_total_engagement_sec,sessions_organic,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,20,0,67,3.350000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,0,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,125,1,616,4.928000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,7,0,28,4.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,11,0,25,2.272727,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Feature Notes

The selected features represent observed performance signals available in the daily warehouse data.

| Feature group | Meaning | Missing values | Available before prediction? |
|---|---|---|---|
| `gsc_impressions` | Search impressions for the content page | Kept as missing when unavailable | Yes, for the observation window |
| `gsc_clicks` | Search clicks for the content page | Kept as missing when unavailable | Yes, for the observation window |
| `gsc_sum_position` | Total search-position signal | Kept as missing when unavailable | Yes, for the observation window |
| `gsc_avg_position` | Average observed search position | Kept as missing when unavailable | Yes, for the observation window |
| `ga4_pageviews` | Page views measured by GA4 | Kept as missing when unavailable | Yes, for the observation window |
| `ga4_sessions` | GA4 sessions | Kept as missing when unavailable | Yes, for the observation window |
| `ga4_users` | GA4 users | Kept as missing when unavailable | Yes, for the observation window |
| `ga4_engaged_sessions` | Engaged GA4 sessions | Kept as missing when unavailable | Yes, for the observation window |
| `ga4_total_engagement_sec` | Total engagement time | Kept as missing when unavailable | Yes, for the observation window |
| `sessions_organic` | Organic search sessions | Kept as missing when unavailable | Yes, for the observation window |
| `sessions_direct` | Direct sessions | Kept as missing when unavailable | Yes, for the observation window |
| `sessions_referral` | Referral sessions | Kept as missing when unavailable | Yes, for the observation window |
| `sessions_social` | Social sessions | Kept as missing when unavailable | Yes, for the observation window |
| `sessions_paid` | Paid sessions | Kept as missing when unavailable | Yes, for the observation window |
| `sessions_ai` | Sessions attributed to AI sources | Kept as missing when unavailable | Yes, for the observation window |
| `ai_chatgpt` to `ai_other` | AI-source-specific sessions | Kept as missing when unavailable | Yes, for the observation window |
| `scroll_events` | Observed scroll events | Kept as missing when unavailable | Yes, for the observation window |

The selected features are numerical performance signals, so no categorical encoding is required for this feature vector.

Missing values are not automatically converted to zero because the ML-04 data contract showed that GSC and GA4 availability differs across observations. A missing value can therefore indicate unavailable measurement rather than zero activity.

The features represent observations from the selected time window. No future post-refresh performance or observed refresh outcome is included.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check feature types, missing values, and basic availability

feature_notes = pd.DataFrame({
    "feature": feature_columns,
    "dtype": [X[col].dtype for col in feature_columns],
    "missing_count": [X[col].isna().sum() for col in feature_columns],
    "missing_percent": [
        round(X[col].isna().mean() * 100, 2)
        for col in feature_columns
    ]
})

print("Feature summary:")
display(feature_notes)

print("\nAll selected features are numeric:")

numeric_check = X[feature_columns].select_dtypes(
    include=["number"]
).columns.tolist()

print(len(numeric_check) == len(feature_columns))

print("\nNumber of numeric features:", len(numeric_check))

Feature summary:


,feature,dtype,missing_count,missing_percent
0,gsc_impressions,int64,0,0.00
1,gsc_clicks,int64,0,0.00
2,gsc_sum_position,int64,0,0.00
3,gsc_avg_position,float64,6230317,63.31
4,ga4_pageviews,float64,3018741,30.67
5,ga4_sessions,float64,3018741,30.67
6,ga4_users,float64,3018741,30.67
7,ga4_engaged_sessions,float64,3018741,30.67
8,ga4_total_engagement_sec,float64,3018741,30.67
9,sessions_organic,float64,3018741,30.67



All selected features are numeric:
True

Number of numeric features: 23


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage Check

I checked the feature vector for three main leakage risks.

**1. Label leakage:**  
The warehouse does not contain an observed `refresh_needed`, `refresh_priority`, or post-refresh outcome field. Therefore, no observed refresh label is included in the features.

**2. Future-window leakage:**  
The selected features come from the observed March 2026 performance window. I do not use later observations or post-refresh performance as input features.

**3. Identifier and context leakage:**  
`client_hash_id` and `content_hash_id` are not included because they identify entities rather than describe their performance. Date and warehouse partition fields are also excluded from the feature vector.

The feature vector therefore contains observed performance signals rather than a label-derived or future outcome field.

This check supports the use of the features for directional, decision-support ranking rather than claiming a measured refresh outcome.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage hunt

print("LEAKAGE HUNT")
print("============")

# 1. Check whether any obvious label/outcome fields exist
possible_label_fields = [
    col for col in df.columns
    if any(term in col.lower()
           for term in ["refresh", "label", "outcome", "priority"])
]

print("\n1. Possible label/outcome fields:")
print(possible_label_fields)


# 2. Check that known label/outcome fields are not in the feature vector
known_outcome_fields = [
    "refresh_needed",
    "refresh_priority",
    "post_refresh_performance",
    "future_clicks",
    "future_impressions"
]

leakage_in_features = [
    col for col in feature_columns
    if col in known_outcome_fields
]

print("\n2. Known outcome fields included in features:")
print(leakage_in_features)


# 3. Check that identifiers/context fields are not features
excluded_context_fields = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "month",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available"
]

context_leakage = [
    col for col in feature_columns
    if col in excluded_context_fields
]

print("\n3. Context/identifier fields included in features:")
print(context_leakage)


# 4. Final check
leakage_check_passed = (
    len(leakage_in_features) == 0
    and len(context_leakage) == 0
)

print("\nOverall leakage check passed:", leakage_check_passed)

LEAKAGE HUNT

1. Possible label/outcome fields:
[]

2. Known outcome fields included in features:
[]

3. Context/identifier fields included in features:
[]

Overall leakage check passed: True


### Excluded Fields

The following fields are excluded from the feature vector:

- `client_hash_id` — identifies the client and does not describe content performance.
- `content_hash_id` — identifies the content page and does not describe its performance.
- `report_date` — used to define and verify the observation window, but not used as a direct performance feature.
- `month` — warehouse partition/context field, not a performance signal.
- `client_has_gsc` — indicates whether GSC is available for the client rather than measuring performance.
- `client_has_ga4` — indicates whether GA4 is available for the client rather than measuring performance.
- `gsc_data_available` — describes whether GSC data is available for the observation.
- `ga4_data_available` — describes whether GA4 data is available for the observation.

There is also no observed `refresh_needed`, `refresh_priority`, or post-refresh outcome field in the warehouse, so no such label is included.

The availability fields remain useful as context because a missing performance value should not automatically be interpreted as zero.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify that excluded fields are not part of the feature vector

excluded_fields = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "month",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available"
]

excluded_from_features = [
    col for col in excluded_fields
    if col not in feature_columns
]

print("Excluded fields:")
for col in excluded_from_features:
    print("-", col)

print("\nNumber of excluded fields:", len(excluded_from_features))

print(
    "\nAll intended excluded fields are outside the feature vector:",
    len(excluded_from_features) == len(excluded_fields)
)

Excluded fields:
- client_hash_id
- content_hash_id
- report_date
- month
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available

Number of excluded fields: 8

All intended excluded fields are outside the feature vector: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.